In [48]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.text_content[1]

'Yes unit whatever bring. Skill something knowledge company entire card wonder. Hard travel employee project.\nSchool road people son answer treat bank. Into economic television man level. Practice human describe.\nWoman issue technology all child compare. Sister middle student question. Born Democrat must reason area.\nLot check service reflect.\nFear bar him minute house. Read might themselves century. Evening machine whom cover interview join.\nChance capital fall wait American include scene. Former full without employee blue show reality.'

In [49]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', "", text)
    text = re.sub(r'[\d+]', "", text)
    text = re.sub("\\n", '', text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return text

df_train.text_content = df_train.text_content.apply(preprocess)
df_test.text_content = df_test.text_content.apply(preprocess)

df_train.head()

,id,text_content,sentiment_score
0,1102,instead wish question treat break billion half...,-0.6707
1,1303,yes unit whatever bring skill something knowle...,0.2156
2,1217,expect improve push voice special billion resu...,0.8601
3,1224,respond role charge body summer audience wife ...,0.6818
4,895,player system quality crime treatment over art...,-0.9436


In [50]:
df_train.text_content[1]

'yes unit whatever bring skill something knowledge company entire card wonder hard travel employee projectschool road people son answer treat bank into economic television man level practice human describewoman issue technology all child compare sister middle student question born democrat must reason arealot check service reflectfear bar him minute house read might themselves century evening machine whom cover interview joinchance capital fall wait american include scene former full without employee blue show reality'

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df_train.text_content
y = df_train.sentiment_score
X_final = df_test.text_content

vect = TfidfVectorizer()
X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [52]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor()
model.fit(X, y)
y_preds = model.predict(X_test)
mean_absolute_error(y_test, y_preds)

0.18137673809523805

In [53]:
y_final = model.predict(X_final)

In [54]:
#from sklearn.manifold import TSNE

#X_final = TSNE(n_components=2, init="random").fit_transform(X_final)

#from sklearn.decomposition import PCA
#X_final = PCA(n_components=50).fit_transform(X_final)

In [55]:
from sklearn.cluster import KMeans

model2 = KMeans(n_clusters=2, random_state=42)
y_final2 = model2.fit_predict(X_final)

In [56]:
subtask_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.id,
    "answer": 50
})

subtask_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.id,
    "answer": y_final
})

subtask_3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.id,
    "answer": y_final2
})

submission = pd.concat([subtask_1, subtask_2, subtask_3])
submission

,subtaskID,datapointID,answer
0,1,383,50.0
1,1,1356,50.0
2,1,945,50.0
3,1,387,50.0
4,1,15,50.0
...,...,...,...
258,3,946,0.0
259,3,742,0.0
260,3,689,0.0
261,3,730,0.0


In [57]:
submission.to_csv("submission.csv", index=False)